# 03 — MLflow Sessions: Multi-Turn Chat Tracking

**UI tab:** Traces → Sessions view

A **session** groups multiple traces that belong to the same conversation. Each turn in the chat creates one trace. Sharing the same `session_id` links them into a thread in the MLflow UI.

```
Session abc-123
├── Turn 1  →  "What is MLflow?"
├── Turn 2  →  "How does tracing work?"
└── Turn 3  →  "What are sessions?"
```

> Start the MLflow server first: `mlflow server --host 127.0.0.1 --port 5000`

In [ ]:
!pip install mlflow google-generativeai --quiet

In [ ]:
import os, uuid
import google.generativeai as genai
import mlflow

os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY_HERE"
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("03-MLflow-Sessions")

print("MLflow", mlflow.__version__, "ready")

## Session 1 — A simple 3-turn conversation

In [ ]:
# Every conversation gets a unique session ID
session_id = f"session-{str(uuid.uuid4())[:8]}"
print(f"Session ID: {session_id}")

model = genai.GenerativeModel(
    "gemini-2.5-flash",
    system_instruction="You are an MLflow expert. Answer in 2 sentences max."
)
chat = model.start_chat(history=[])  # Gemini keeps conversation memory automatically

turns = [
    "What is MLflow?",
    "What is the Traces tab in MLflow?",
    "What is an MLflow session?"
]

with mlflow.start_run(run_name=f"chat-{session_id}"):
    mlflow.log_param("session_id", session_id)

    for i, user_msg in enumerate(turns, start=1):
        # Each turn = one span tagged with the session_id
        with mlflow.start_span(name=f"turn-{i}", span_type="CHAT_MODEL") as span:
            span.set_attribute("mlflow.session_id", session_id)
            span.set_attribute("user", user_msg)

            response = chat.send_message(user_msg)
            answer = response.text.strip()

            span.set_attribute("assistant", answer)

        print(f"[Turn {i}] User: {user_msg}")
        print(f"         Bot:  {answer}\n")

## Session 2 — A second conversation on a different topic

In [ ]:
session_id_2 = f"session-{str(uuid.uuid4())[:8]}"
print(f"Session ID: {session_id_2}")

model2 = genai.GenerativeModel(
    "gemini-2.5-flash",
    system_instruction="You are a Python tutor. Answer in 2 sentences max."
)
chat2 = model2.start_chat(history=[])

turns2 = [
    "What is a Python list?",
    "How is it different from a tuple?"
]

with mlflow.start_run(run_name=f"chat-{session_id_2}"):
    mlflow.log_param("session_id", session_id_2)
    mlflow.set_tag("topic", "python-basics")

    for i, user_msg in enumerate(turns2, start=1):
        with mlflow.start_span(name=f"turn-{i}", span_type="CHAT_MODEL") as span:
            span.set_attribute("mlflow.session_id", session_id_2)
            span.set_attribute("user", user_msg)

            response = chat2.send_message(user_msg)
            answer = response.text.strip()
            span.set_attribute("assistant", answer)

        print(f"[Turn {i}] User: {user_msg}")
        print(f"         Bot:  {answer}\n")

## MLflow UI — What to explore
```
Traces tab → Sessions view
├── session-<id1>   →  3 turns on MLflow topics
└── session-<id2>   →  2 turns on Python

Click any session → see the full conversation thread
Click any turn → see user message + assistant response
```
**Next →** `04_mlflow_judges_evaluation.ipynb`